In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
from ep_seq import NMTDataset, GO_token, EOS_token, PAD_token

SEQ_MXLEN= 16  # max sequence length

HIDDEN_N= 64
BATCH_SIZE= 128  # the implementation processes the sequence element by element

DATASET_SIZE = BATCH_SIZE*(131072//BATCH_SIZE)

Dataset_nmt = NMTDataset('EP_datasets/eng-fra.txt', DATASET_SIZE, SEQ_MXLEN, vectorized=True)

# sanity
print(f"#seqs={len(Dataset_nmt)} {Dataset_nmt.input_lang.name} to {Dataset_nmt.output_lang.name}")
print(f"#voc= {Dataset_nmt.input_lang.n_words} to {Dataset_nmt.output_lang.n_words}")

seq1d, seq2d = Dataset_nmt.dataset[0][0].view(-1).tolist(), Dataset_nmt.dataset[0][1].view(-1).tolist()
seq1s, seq2s = [Dataset_nmt.input_lang.ix2word[_] for _ in seq1d], [Dataset_nmt.output_lang.ix2word[_] for _ in seq2d]

print(f"{seq1d} --- {seq2d}")
print(f"{seq1s} --- {seq2s}")

#seqs=131072 fra to eng
#voc= 4966 to 3240
[3, 4, 5, 6, 7, 8, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2] --- [3, 4, 5, 6, 7, 8, 9, 1, 2, 2, 2, 2, 2, 2, 2, 2]
['j', 'essaye', 'd', 'arreter', 'tom', '.', 'EOS', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD'] --- ['i', 'm', 'trying', 'to', 'stop', 'tom', '.', 'EOS', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']


In [5]:
class Encoder(nn.Module):
    def __init__(self, n_input:int, n_hidden:int):
        super().__init__()
        self.n_input, self.n_hidden = n_input, n_hidden
        # embedding layer - n_input=vocabulary size, n_hidden=will be embedding size
        self.embedding = nn.Embedding(self.n_input, self.n_hidden, padding_idx=PAD_token)
        # RNN layer, used token by token
        self.rnn_cell = nn.GRU(self.n_hidden, self.n_hidden, batch_first=True)

    # BLUE in the block diagram
    def forward(self, _X, _hn):
        # BS x T x hidden_size
        X_embedded = self.embedding(_X)
        output, hn = self.rnn_cell(X_embedded, _hn)  # RNN iterations inside
        return output, hn

    def init_hidden(self, bs=BATCH_SIZE):  # 1 x BS x H by design of PyTorch GRU, or T x BS x H
        return torch.zeros(1, bs, self.n_hidden, device=Device)

class Decoder(nn.Module):
    def __init__(self, n_hidden:int, n_output:int, dropout:float=0.1):
        super().__init__()
        self.n_hidden, self.n_output = n_hidden, n_output

        self.embedding = nn.Embedding(self.n_output, self.n_hidden, padding_idx=PAD_token)
        self.dropout = nn.Dropout(dropout)
        self.attn = nn.Linear(in_features=self.n_hidden, out_features=self.n_hidden)  # W_a
        # attention outputs and hn will be cascaded
        self.w_c = nn.Linear(in_features=2*self.n_hidden, out_features=self.n_hidden)  # combine [h_t; c_t]
        self.rnn_cell = nn.GRU(input_size=self.n_hidden, hidden_size=self.n_hidden, batch_first=True)

        # output
        self.w_y = nn.Linear(in_features=self.n_hidden, out_features=self.n_output)

    def forward(self, _X, _hn, _encoder_outputs, enc_len):
        # BS x T x hidden_size
        X_embedded = self.embedding(_X)
        X_embedded = self.dropout(X_embedded)

        # RED in the block diagram
        # hidden state, time t
        rnn_out, hn = self.rnn_cell(X_embedded, _hn)

        # YELLOW in the block diagram, alignment_scores shape:Bx1xT
        alignment_scores = torch.bmm(self.attn(hn.permute(1,0,2)), _encoder_outputs.transpose(1,2))

        # weights, shape:Bx1xT
        attn_weights = nn.functional.softmax(alignment_scores, dim=2)

        # multiplicative attention context vector c_t, shape:Bx1xH
        c_t = torch.bmm(attn_weights, _encoder_outputs)

        # concatenate h_t and the context, shape:Bx1x2H
        hidden_s_t = torch.cat([hn.permute(1,0,2), c_t], dim=2)

        # RED in the block diagram
        # hidden context, shape:Bx1xH
        hidden_s_t = torch.tanh(self.w_c(hidden_s_t))

        # output
        output = nn.functional.log_softmax(self.w_y(hidden_s_t), dim=2)
        # shapes output, hn, attn_w: Bx1xn_output, Bx1xH, Bx1xT
        return output, hn, attn_weights

    def init_hidden(self, bs=BATCH_SIZE):  # 1 x BS x H by design of PyTorch GRU, or T x BS x H
        return torch.zeros(1, bs, self.n_hidden, device=Device)

In [6]:
encoder = Encoder(Dataset_nmt.input_lang.n_words, HIDDEN_N).to(Device)
decoder = Decoder(HIDDEN_N, Dataset_nmt.output_lang.n_words, dropout=0.1).to(Device)

Dloader_nmt = DataLoader(Dataset_nmt, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

In [7]:
def train(_encoder, _decoder, epochs=3, info=False):
    import sys

    s2s_loss_func = nn.NLLLoss(ignore_index=PAD_token)  # needs (B,C) tensor shape
    e_optim = torch.optim.AdamW(_encoder.parameters(), lr=5e-4)
    d_optim = torch.optim.AdamW(_decoder.parameters(), lr=5e-4)

    _encoder.train()
    _decoder.train()
    N = len(Dloader_nmt)
    totloss = 0
    for e in range(epochs):
        for b, (seq1, seq2) in enumerate(Dloader_nmt):
            seq1 = seq1.to(Device, non_blocking=True)  # x_t
            seq2 = seq2.to(Device, non_blocking=True)  # y_t

            e_hn = _encoder.init_hidden()  # hidden layer - latent space

            e_optim.zero_grad()
            d_optim.zero_grad()

            Nseq1, Nseq2 = seq1.size(1), seq2.size(1)  # input and target lengths

            # Generate BLUE
            e_outputs = torch.zeros(BATCH_SIZE, SEQ_MXLEN, _encoder.n_hidden, device=Device)

            with torch.set_grad_enabled(True):
                # encoder
                for ei in range(Nseq1):
                    e_output, e_hn = _encoder(seq1[:,ei].unsqueeze(1), e_hn)
                    e_outputs[:,ei] = e_output[:, 0]

                # init decoder with GO_token
                d_input = torch.full((BATCH_SIZE, 1), GO_token, device=Device)

                d_hn = e_hn  # hidden layer connection between encoder and decoder, red thick line in the figure

                d_outputs = []

                for di in range(Nseq2):
                    d_output, d_hn, _ = _decoder(d_input, d_hn, e_outputs, Nseq1)
                    d_outputs.append(d_output.squeeze(1))
                    d_input = seq2[:,di].unsqueeze(1)
                    # loss input (B,C) net-output & token ix

                d_outputs = torch.stack(d_outputs, dim=1)  # B x T x n_output
                loss = s2s_loss_func(d_outputs.view(-1, _decoder.n_output), seq2.view(-1))  # B

                loss.backward()
                e_optim.step()
                d_optim.step()

            totloss += loss.item()

            if info:
                sys.stderr.write(f"\r{b:4d}/{N:4d} | Loss: {totloss:3.2f}")
                sys.stderr.flush()
                totloss = 0

In [ ]:
%%time

train(encoder, decoder, epochs=10, info=True)

In [8]:
def eval_ss(_enc, _dec, _seq1, max_length=SEQ_MXLEN):
    _enc.eval()
    _dec.eval()
    with torch.no_grad():
        Nseq1 = _seq1.size(0)
        _seq1 = _seq1.to(Device)
        e_hn = _enc.init_hidden(bs=1)
        e_outputs = torch.zeros(1, max_length, _enc.n_hidden, device=Device)

        for ei in range(Nseq1):
            e_output, e_hn = _enc(_seq1[ei].view(1,1), e_hn)
            e_outputs[0,ei] = e_output[0, 0]

        d_input = torch.full((1, 1), GO_token, device=Device)

        d_hn = e_hn  # hidden layer connection between encoder and decoder

        d_words = []
        d_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            d_output, d_hn, d_attn = _dec(d_input, d_hn, e_outputs, Nseq1)
            d_attentions[di] = d_attn.data[0]

            # output word index with highest P
            _, topi = d_output.data[0].topk(1)
            if topi.item() != EOS_token:
                d_words.append(Dataset_nmt.output_lang.ix2word[topi.item()])
            else:
                break

            d_input = topi.detach()

        return d_words, d_attentions[:di+1, :Nseq1]  # crop to true encoder length


def eval_rnd(_enc, _dec, n=10):
    import random
    for _ in range(n):
        sample = random.randint(0, len(Dloader_nmt)-1)
        pair = Dataset_nmt.pairs[sample]
        seq1 = Dataset_nmt[sample][0]

        output_words, attentions = eval_ss(_enc, _dec, seq1)

        print(f"seq1:{pair[0]} | seq2:{pair[1]} | result:{' '.join(output_words)}")

In [9]:
eval_rnd(encoder, decoder, 20)

seq1:c est loin d etre un gentleman . | seq2:he is anything but a gentleman . | result:slowing course typist typist baker typist savings bench idea plastered rarely psyched brown web evicted typist
seq1:il est assez vieux pour etre son grand pere . | seq2:he s old enough to be her grandfather . | result:slowing agreement others ve mobile aids cutting attending science mountains lean scientist asleep oceanographer proposal war
seq1:je descends l escalier . | seq2:i am going down the stairs . | result:slowing course typist typist typist savings bench response tallest unemployed unemployed saving unemployed saving plane attending
seq1:il est en excellente condition physique . | seq2:he is in excellent physical condition . | result:slowing biased asleep dog typist typist savings train typist mom allergic brown PAD suddenly fastest fastest
seq1:je suis etudiant . | seq2:i am a student . | result:slowing agreement journalist singing receding attending country oceanographer proposal war court

In [11]:
eval_rnd(encoder, decoder, 20)

seq1:il est toujours cloue au lit . | seq2:he s still sick in bed . | result:slowing course typist typist baker presents agreement how suddenly fastest fastest fastest fastest fastest physical fastest
seq1:nous sommes timides . | seq2:we re shy . | result:slowing course typist typist typist savings bench response tallest slowing agreement gentleman sinking soft fastest fastest
seq1:je me demande si je l aime . | seq2:i m wondering if i love him . | result:slowing course typist typist typist savings bench france terms outgoing daily france fastest fastest fastest fastest
seq1:nous sommes tres heureux . | seq2:we re very happy . | result:slowing course typist typist typist mom allergic attending unaccounted mountains boiling could orders saving typist savings
seq1:je suis desolee de t avoir derange ! | seq2:i m sorry to have bothered you . | result:slowing agreement something timer slowing agreement criticizing noise sometimes character receding terrified attending miss hardly lean
seq1:

In [12]:
eval_rnd(encoder, decoder, 20)

seq1:tu es tres vive . | seq2:you re very sharp . | result:slowing course typist typist typist savings response tallest unemployed saving unemployed saving wisdom typist typist savings
seq1:je perds patience avec vous . | seq2:i am losing my patience with you . | result:slowing course typist typist typist savings response tallest slowing agreement journalist aids polite girls those receding
seq1:je ne suis pas sur de vouloir le voir . | seq2:i m not sure i want to see it . | result:slowing agreement polite fail testing receding america tallest slowing agreement criticizing noise sometimes character receding terrified
seq1:je n en ai pas fini avec toi . | seq2:i m not finished with you . | result:slowing wanting karate courteous flying courteous humid aids toothache travel clothing impress village terms homeschooled could
seq1:elle parle couramment l anglais et le francais . | seq2:she is fluent in english and french . | result:slowing poet could scientists volunteering agreement misera

In [13]:
eval_rnd(encoder, decoder, 20)

seq1:je compte sur vous pour prononcer le discours inaugural . | seq2:i am counting on you to give the opening address . | result:slowing biased asleep agreement journalist asleep dog typist courageous super attending hardly typist typist savings asleep
seq1:tu es precisement la personne a laquelle je veux parler . | seq2:you re just the person i want to speak to . | result:slowing biased grilling cutting son dieting skeleton disaster t years determined tallest slowing agreement something jolly
seq1:vous etes tres genereuse . | seq2:you re very generous . | result:slowing course typist typist typist savings bench response tallest lousy agreement beer fastest fastest fastest fastest
seq1:vous etes pour ainsi dire un poisson hors de l eau . | seq2:you are so to speak a fish out of water . | result:slowing biased asleep jolly typist typist savings typist mom wealth starving wealth allow character receding attending
seq1:je n en peux plus de toutes ces disputes . | seq2:i m sick and tired 

In [10]:
def params_count(_net):
    return sum(p.numel() for p in _net.parameters() if p.requires_grad)

# total size of the transformer
print(f"Num Params= {params_count(encoder)+params_count(decoder):,}")

Num Params= 798,120
